In [ ]:
# --- Tutorial setup ---
# This cell loads everything needed to run this notebook standalone.
# It is hidden in the rendered documentation but visible when running the
# notebook as a tutorial.
import os
import sys
sys.path.insert(0, "/".join(["..", "python"]))

import warnings
import numpy as np
import rasterio
warnings.filterwarnings("ignore", category=rasterio.errors.NotGeoreferencedWarning)

from gridr.chain.grid_resampling_chain import basic_grid_resampling_chain
from gridr.misc.mandrill import mandrill
from notebook_utils import plot_im
from chain_notebook_utils import (
    write_array, create_grid, create_star_polygon, plot_grid_on_image,
)

IN_DOC_BUILD = os.environ.get("DOC_BUILD", "0") == "1"
if not IN_DOC_BUILD:
    from bokeh.io import output_notebook
    output_notebook()

# File paths shared across the chain tutorial series.
RASTER_IN          = "./grid_resampling_chain_raster_in.tif"
GRID_IN_F64        = "./grid_resampling_chain_grid_in_f64.tif"
output_raster_path = "./grid_resampling_chain_output_raster.tif"
output_mask_path   = "./grid_resampling_chain_output_mask.tif"

# Grid parameters shared across the chain tutorial series.
nrow, ncol = 50, 40
origin_pos  = np.array((0.3, 0.2))
origin_node = np.array((0., 0.))
v_row_y, v_row_x = 5.2, 1.2
v_col_y, v_col_x = -2.7, 7.1

grid_row_f64, grid_col_f64 = create_grid(
    nrow, ncol, origin_pos, origin_node,
    v_row_y, v_row_x, v_col_y, v_col_x, dtype=np.float64,
)

# Hybrid input creation: ensure the source raster and grid file exist on disk.
if not os.path.exists(RASTER_IN):
    write_array(mandrill, dtype=mandrill.dtype, fileout=RASTER_IN)
if not os.path.exists(GRID_IN_F64):
    write_array(np.array([grid_row_f64, grid_col_f64]),
                dtype=np.float64, fileout=GRID_IN_F64)

# Default output-dataset open arguments (resolution-dependent overrides
# are applied per-notebook below).
grid_resolution = (8, 8)
from gridr.core.grid.grid_commons import grid_full_resolution_shape
output_shape = grid_full_resolution_shape(
    shape=grid_row_f64.shape, resolution=grid_resolution,
)
raster_out_open_args = {
    "driver": "GTiff", "dtype": np.float64,
    "height": output_shape[0], "width": output_shape[1], "count": 1,
}
mask_out_open_args = {
    "driver": "GTiff", "dtype": np.uint8,
    "height": output_shape[0], "width": output_shape[1],
    "count": 1, "nbits": 1,
}

grid_mask_in_path        = "./grid_resampling_chain_grid_mask_in_u8_1_0.tif"
grid_mask_in_path_i8     = "./grid_resampling_chain_grid_mask_in_i8_0_m10.tif"
raster_mask_in_path_u8   = "./grid_resampling_chain_raster_mask_in_u8_1_0.tif"

# Ensure the grid mask exists on disk (created in notebook 003 if missing).
grid_mask_in_valid_value, grid_mask_in_invalid_value = 1, 0
roi = np.array(((10, 40), (5, 100)))
grid_mask = np.full(grid_row_f64.shape, grid_mask_in_valid_value, dtype=np.uint8)
grid_mask[
    np.logical_and(
        np.logical_and(grid_row_f64 >= roi[0][0], grid_row_f64 <= roi[0][1]),
        np.logical_and(grid_col_f64 >= roi[1][0], grid_col_f64 <= roi[1][1]),
    )
] = grid_mask_in_invalid_value
if not os.path.exists(grid_mask_in_path):
    write_array(grid_mask, dtype=np.uint8, fileout=grid_mask_in_path)

# Ensure the source raster mask exists on disk (created in notebook 004 if missing).
array_src_mask_validity_valid, array_src_mask_validity_invalid = 1, 0
array_in_mask = np.full(mandrill[0].shape, array_src_mask_validity_valid, dtype=np.uint8)
array_in_mask[slice(50, 71), slice(150, 171)] = array_src_mask_validity_invalid
if not os.path.exists(raster_mask_in_path_u8):
    write_array(array_in_mask, dtype=np.uint8, fileout=raster_mask_in_path_u8)


# I/O and memory management

By default the chain processes the entire output in a single pass. When the raster, grid or output dimensions become large, this approach can exceed available memory. Three parameters let you trade memory against the number of I/O operations: `io_strip_size`, `io_strip_size_target` and `tile_shape`.

**What you'll learn**

- The chain's main loop: strips and tiles
- How `io_strip_size` and `io_strip_size_target` control row stripping
- How `tile_shape` further subdivides each strip
- How to enable debug logging via the `logger` argument
- The chain's separate `grid_ds` / `grid_col_ds` arguments

## The main loop

`basic_grid_resampling_chain` processes the output raster in sequential row strips. For each strip, it computes all the output columns (or the subset defined by `win`) of a contiguous range of rows. The maximum strip height is `io_strip_size`.

`io_strip_size_target` selects how that height is interpreted:
- `GridRIOMode.OUTPUT` (default): the height is expressed in **output**   rows.
- `GridRIOMode.INPUT`: the height is expressed in **input grid** rows   and scaled by `grid_resolution`.

Setting `io_strip_size = 0` disables stripping (the whole output is processed at once).

At each iteration the chain reads only the source-raster region needed for the current output strip. Two reusable buffers are pre-allocated (input grid data, output raster data), sized to the largest strip.

## Internal tiling

`tile_shape` further divides each strip into `(nrows, ncols)` tiles. For each tile, the chain calls `array_compute_resampling_grid_geometries` to determine the minimal bounding box of source pixels required, then reads only that region. Tiles entirely outside the source domain or fully masked by the grid mask are skipped.

Tiling is most beneficial when the grid is diagonal relative to the source raster, since rectangular reads otherwise pull in large amounts of unused data. Tiles do **not** share memory for the input data, so the same source area may be read several times; very small tiles cause redundant reads. Setting `tile_shape = None` disables tiling.

## A note on the *basic* prefix

The `basic_` prefix marks the lack of automatic parameter tuning. Optimal `io_strip_size` and `tile_shape` values depend on the actual grid coordinates, not on the chain parameters themselves. Future versions may automate this; for now, set the parameters based on what you know about your grid."),


## Configuring a debug logger

Passing a `logging.Logger` configured at `DEBUG` level through the `logger` argument enables verbose tracing of the strip/tile iterations -- useful when fine-tuning the parameters.

In [ ]:
import logging
from gridr.io.common import GridRIOMode

log_file = "./gridr_resampling_chain_007_gridr_log.log"
fhandler  = logging.FileHandler(filename=log_file, mode="w")
fhandler.setFormatter(logging.Formatter(
    "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
))
gridr_logger = logging.getLogger("gridr")
gridr_logger.setLevel(logging.DEBUG)
gridr_logger.addHandler(fhandler)

## Running with explicit stripping and tiling

Below we cap the output strip at 200 rows (giving 2 strips for a 393-row output) and divide each strip into `(200, 160)` tiles.

In [ ]:
with rasterio.open(GRID_IN_F64, "r") as grid_in_ds, \
        rasterio.open(RASTER_IN, "r") as array_src_ds, \
        rasterio.open(grid_mask_in_path, "r") as grid_mask_in_ds, \
        rasterio.open(raster_mask_in_path_u8, "r") as raster_mask_in_ds, \
        rasterio.open(output_raster_path, "w", **raster_out_open_args) as array_out_ds, \
        rasterio.open(output_mask_path, "w", **mask_out_open_args) as mask_out_ds:

    basic_grid_resampling_chain(
        grid_ds                        = grid_in_ds,
        grid_row_coords_band           = 1,
        grid_col_coords_band           = 2,
        grid_resolution                = grid_resolution,
        array_src_ds                   = array_src_ds,
        array_src_bands                = 1,
        array_out_ds                   = array_out_ds,
        interp                         = "cubic",
        nodata_out                     = 0,
        mask_out_ds                    = mask_out_ds,
        grid_mask_in_ds                = grid_mask_in_ds,
        grid_mask_in_unmasked_value    = grid_mask_in_valid_value,
        grid_mask_in_band              = 1,
        array_src_mask_ds              = raster_mask_in_ds,
        array_src_mask_band            = 1,
        array_src_mask_validity_pair   = (array_src_mask_validity_valid,
                                          array_src_mask_validity_invalid),
        # I/O controls:
        io_strip_size        = 200,
        io_strip_size_target = GridRIOMode.OUTPUT,
        tile_shape           = (200, 160),
        logger               = gridr_logger,
    )

The log file produced is shown below (scrollable). Each line traces a strip / tile iteration with the source read window it required.

In [ ]:
from IPython.display import display, Markdown
import html

try:
    with open(log_file) as f:
        content = html.escape(f.read())
except FileNotFoundError:
    content = "Log file not found."

display(Markdown(
    '''<div style="max-height:300px;overflow-y:auto;border:1px solid #ccc;'''
    '''font-family:monospace;font-size:11px;padding:6px;white-space:pre">'''
    + content + "</div>"
))

## Separate datasets for row and column grids

`basic_grid_resampling_chain` accepts `grid_ds` for row coordinates and an optional `grid_col_ds` for column coordinates. When `grid_col_ds` is `None` (default) or the same object as `grid_ds`, band 1 holds row coordinates and band 2 holds column coordinates in the same file -- the convention used throughout this tutorial series. Providing two distinct files is useful when the grid is stored split across separate rasters.